# Phase 5 — a second forecasting dataset, and the trap it came with

Every claim this project has made about **ordered splits** rests on one dataset. Rossmann
told us that the split has to respect time (0043), that a frozen history is anchored at a
single date and that date is a lottery (0049, 0050), and that a trailing window's measured
worth is mostly an artifact of the fold count (0053). All of it is one reading.

So: **Recruit Restaurant Visitor Forecasting**. Same *shape* of problem — one row per
entity per day, forecast a block of later dates — at a twentieth of the size, with a
different metric and a genuinely multi-file join.

You'll do the join by hand first, exactly as `scripts/prep_restaurant.py` does it. Then
you'll find the thing that makes this dataset worth having, which is not the join.

> **Datasets are probes, not targets.** We are not here to score well on restaurants. We
> are here to find out what the harness cannot express. This one found something within an
> hour.

## 1. Six files, one grain

The harness takes **one table** and has no join tool. That is deliberate: a join is a
decision about *grain*, not a feature transform, and the harness does not make modelling
decisions for the agent (0029).

So the join happens once, in the open, in `scripts/prep_*.py`. Here are the pieces.

In [ ]:
import os, sys
import pandas as pd

REPO = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))
sys.path.insert(0, os.path.join(REPO, "scripts"))
RAW = os.path.join(REPO, "restaurant_visitor")

for name in sorted(os.listdir(RAW)):
    if name.endswith(".csv"):
        df = pd.read_csv(os.path.join(RAW, name), nrows=2)
        n = sum(1 for _ in open(os.path.join(RAW, name))) - 1
        print(f"{name:<26} {n:>8,} rows   {list(df.columns)}")

Three different grains sit in that list, and the difference is the whole job:

| file | one row is | joins how |
|---|---|---|
| `air_visit_data` | a **store-day** | the spine — this *is* the table |
| `air_store_info` | a **store** | `many_to_one` on the store id |
| `date_info` | a **date** | `many_to_one` on the date |
| `air_reserve` | a **reservation** | finer than the spine — must be **aggregated** first |
| `hpg_reserve` | a **reservation**, keyed differently | aggregate *and* translate the key |
| `store_id_relation` | the key translation | 150 of 829 stores |

The two reservation files are the interesting ones. There is no harness tool that takes a
second file at a finer grain — `fit_history` aggregates *within* a table — so this step
could not have happened inside a run even if we wanted it to.

In [ ]:
visits = pd.read_csv(os.path.join(RAW, "air_visit_data.csv"), parse_dates=["visit_date"])
print(visits.shape, "|", visits.air_store_id.nunique(), "stores")
print(visits.visit_date.min().date(), "->", visits.visit_date.max().date())
visits.head()

### The spine is ragged, and that is a fact worth holding on to

Rossmann has a row for every store on every day, including days the store was shut
(`Sales == 0`, 17% of rows). This dataset does **not**: a store with no row on a date
simply did not report it.

Count the days each store actually has, against the days it *could* have had.

In [ ]:
span = (visits.visit_date.max() - visits.visit_date.min()).days + 1
per_store = visits.groupby("air_store_id").agg(
    days_reported=("visit_date", "size"),
    first=("visit_date", "min"),
    last=("visit_date", "max"),
)
per_store["days_available"] = (per_store["last"] - per_store["first"]).dt.days + 1
per_store["coverage"] = per_store.days_reported / per_store.days_available

print(f"calendar span: {span} days")
print(per_store[["days_reported", "coverage"]].describe().round(2))
print(f"\nstores present from day 1: {(per_store['first'] == visits.visit_date.min()).sum()} of {len(per_store)}")

Half the stores are missing a fifth of their own days, and most of them do not start at
the beginning of the file at all — they appear when they join the platform.

This matters for us specifically. Every window feature in this harness (0052, 0053) counts
*rows*, not days. On Rossmann "the last 364 rows" and "the last 364 days" were the same
thing per store. **Here they are not.** A trailing window of 364 rows can reach back two
calendar years for a store that reports twice a week.

That is not a defect in the dataset. It is the assumption the Rossmann work never had to
notice, and now there is a second reading that will notice it.

## 2. Doing the join by hand

Start with the two easy ones. Note `validate=` on every merge — a join that silently
duplicates rows is the failure mode here, and pandas will refuse rather than let it
happen.

In [ ]:
store = pd.read_csv(os.path.join(RAW, "air_store_info.csv"))
dates = pd.read_csv(os.path.join(RAW, "date_info.csv"), parse_dates=["calendar_date"])

out = visits.merge(store, on="air_store_id", how="left", validate="many_to_one")
out = out.merge(dates, left_on="visit_date", right_on="calendar_date",
                how="left", validate="many_to_one").drop(columns=["calendar_date"])

assert len(out) == len(visits), "the join changed the row count"
out.head(3)

Now the reservations, which need collapsing onto the spine's grain first. Three statistics
— how many people, how many bookings, how far ahead. Not just the mean: 0046 is the
standing warning that a mean alone throws the group away.

In [ ]:
res = pd.read_csv(os.path.join(RAW, "air_reserve.csv"),
                  parse_dates=["visit_datetime", "reserve_datetime"])
res["visit_date"] = res.visit_datetime.dt.normalize()
res["lead_days"] = (res.visit_date - res.reserve_datetime.dt.normalize()).dt.days

print(res.shape, "reservations for", res.air_store_id.nunique(), "stores")
res.head(3)

In [ ]:
air_agg = (res.groupby(["air_store_id", "visit_date"])
              .agg(air_reserve_visitors=("reserve_visitors", "sum"),
                   air_reserve_count=("reserve_visitors", "size"),
                   air_reserve_lead_days=("lead_days", "mean"))
              .reset_index())

out = out.merge(air_agg, on=["air_store_id", "visit_date"], how="left", validate="one_to_one")
print(f"rows with a reservation: {out.air_reserve_count.notna().mean():.1%}")
out[out.air_reserve_count.notna()].head(3)

The hpg file is keyed by a *different* store identifier. `store_id_relation.csv` is the
only bridge, and it covers 150 of 829 stores.

Order matters: translate the key **first**, then aggregate. Aggregating first would leave
the result at the hpg grain, and two hpg ids mapping to one air id would then give two
rows to join against one — which `validate="one_to_one"` would catch, but only after you
had written it the wrong way round.

In [ ]:
rel = pd.read_csv(os.path.join(RAW, "store_id_relation.csv"))
hpg = pd.read_csv(os.path.join(RAW, "hpg_reserve.csv"),
                  parse_dates=["visit_datetime", "reserve_datetime"])
print(f"hpg reservations: {len(hpg):,}")

hpg = hpg.merge(rel, on="hpg_store_id", how="inner", validate="many_to_one")
print(f"  ...that map onto an air store: {len(hpg):,}")

## 3. The trap

Rossmann had `Customers`: correlated 0.895 with the target, absent from the test file, and
a leak that every guard in this harness permits. `prep_rossmann.py` keeps it on purpose
and says so in the data dictionary, because *whether to use a column* is the agent's
decision (0029).

This dataset's trap is subtler, and I did not see it until I checked the timestamps.

The reservation columns you just built **do exist at prediction time** — the competition
ships reservations for the forecast window. So there is no missing column to notice. The
question is whether the column is *made the same way* on both sides.

In [ ]:
FORECAST_START = pd.Timestamp("2017-04-23")
train_res = res[res.visit_date < FORECAST_START]
fcst_res  = res[res.visit_date >= FORECAST_START]

print(f"last reservation ever recorded: {res.reserve_datetime.max()}")
print()
print(f"{'':<22}{'n':>8}{'min lead':>10}{'share booked same day':>24}")
for label, part in [("training period", train_res), ("forecast window", fcst_res)]:
    print(f"{label:<22}{len(part):>8,}{part.lead_days.min():>10}"
          f"{(part.lead_days == 0).mean():>24.1%}")

There it is.

**The reservation stream is cut at 2017-04-22 23:00 — the moment the forecast is issued.**

A training row can count a table booked the same morning, and 30% of training reservations
were booked with zero days' notice. **No row in the forecast window can, ever.** And it
gets worse the further out you forecast: a visit on the last day of the window can only
carry bookings made 39+ days ahead.

In [ ]:
# The *structural* bound — what a booking could possibly have had — next to what
# actually turned up in the file. The structural one is the claim; the observed
# one is noisier because nobody happened to book at exactly that lead.
window = pd.date_range(FORECAST_START, "2017-05-31")
bound = pd.DataFrame({
    "forecast_date": window,
    "min_possible_lead": (window - res.reserve_datetime.max().normalize()).days,
})
bound["min_observed_lead"] = bound.forecast_date.map(
    fcst_res.groupby("visit_date").lead_days.min()
)
print(bound.head(3).to_string(index=False))
print("...")
print(bound.tail(3).to_string(index=False))

### Why this one is worth the dataset

Think about what a guard would have to notice to catch this.

- No fold boundary is crossed — the aggregate for a row uses only that row's own date.
- No encoder is misfitted — there is no encoder.
- The column is present at serving time, so 0030's servability check passes.
- **Validation confirms the column is excellent**, because the validation half is drawn
  from the same uncut stream as training.

Every guard we have says fine. The column is still systematically larger in training than
at serving time, by an amount that *grows with the forecast horizon*.

That is 0047's finding — **the horizon is a column, not a constant** — arriving in a
*feature* instead of in a lag. And the capability it asks for does not exist:

> aggregate a related stream restricted to what was knowable a given lead time ahead

Note the phrasing. No dataset is named. That is the test CLAUDE.md sets for whether a gap
is ours to fix, and this one passes it — "a related stream that keeps arriving after the
forecast is issued" is not a restaurant problem, it is what every reservation, order
book, pre-registration and pipeline column looks like.

**We are not fixing it here.** The column ships as-is and the data dictionary states the
fact plainly, exactly as `Customers` does. If the agent trains on it happily, we learn
something about the dictionary channel that a pre-cleaned table could not have taught us.

## 4. The prepared table

Everything above is what `scripts/prep_restaurant.py` does. Run it and read what it says.

In [ ]:
from prep_restaurant import ABOUT, COLUMN_NOTES, METRIC, OUT, SPLIT_ON, TEST_OUT

work = pd.read_parquet(OUT)
test = pd.read_parquet(TEST_OUT)
print(f"train {work.shape}   test {test.shape}")
print(f"metric {METRIC} | split_on {SPLIT_ON}")
print(f"in train and not in test: {sorted(set(work.columns) - set(test.columns))}")
work.head(3)

In [ ]:
print(ABOUT)
print()
print(COLUMN_NOTES["air_reserve_visitors"])

Read that dictionary entry against the rule it has to satisfy: it states **what the data
is**, not what a modeller should conclude. "The column exists at prediction time; it is
not made the same way there" is a fact about the file. "Therefore drop it" would be the
agent's job, done for it.

The same discipline is why `air_area_name` is left as one string — splitting the
prefecture off is a feature decision — and why no reservation column is filled. Missing
means no reservation was recorded, which is *usually* a true zero and, for the 515 stores
absent from the reservation file entirely, closer to "this store does not take bookings".
Distinguishing those is imputation, and the harness has tools for it.

## 5. Handing it to the harness

Nothing new here — the point is that after the prep script, there *is* nothing new. Same
`RunSession`, same ordered split, same probe.

In [ ]:
import shutil, tempfile
from tabular_agent.harness.session import RunSession
from tabular_agent.taskspec import TaskSpec
from tabular_agent.io import load_table

run_dir = os.path.join(tempfile.mkdtemp(prefix="restaurant_"), "run")
serving = [c for c in test.columns if c != "id"]

session = RunSession(
    run_dir, OUT,
    TaskSpec(target="visitors", metric=METRIC, notes=ABOUT,
             column_notes=COLUMN_NOTES, forecast_horizon=1.0),
    val_fraction=0.05, holdout_fraction=0.05, split_on=SPLIT_ON,
    serving_columns=serving,
)
spec = session.resolved()
print(f"task {spec.task_type} | metric {spec.metric} | target_transform {spec.target_transform}")
print(f"work {len(load_table(session.data_path)):,} | val {len(load_table(session.val_path)):,}")

`val_fraction=0.05` is Rossmann's number, not the library's 0.2 default. The forecast
window is 39 days against 476 of history — about 8% — so a 5% validation block is the
closest this split gets to "one horizon", and matching Rossmann is what makes the two
datasets comparable at all (0043 is why the fraction is load-bearing on ordered data).

`forecast_horizon=1.0` because the window opens the day after training ends. That is the
*floor* across the block; rows deeper in are 2 to 39 days out, and 0047 is the record of
why that is a column rather than a constant.

In [ ]:
work_half = load_table(session.data_path)
val_half = load_table(session.val_path)
print(f"work: {work_half.visit_date.min().date()} -> {work_half.visit_date.max().date()}")
print(f"val : {val_half.visit_date.min().date()} -> {val_half.visit_date.max().date()}")
print(f"\nno overlap: {work_half.visit_date.max() < val_half.visit_date.min()}")

In [ ]:
from tabular_agent.harness.tools import probe_table

base = probe_table(session, session.data_path)
print(f"baseline probe ({METRIC}): {base['probe_score']}")
print(f"  fitted in {base['probe_fit_seconds']}s on transform {base['probe_target_transform']}")

That is the ruler's reading on raw columns with nothing engineered — the number every
later table on this dataset gets compared against.

Which raises the question the next notebook is about: **is the ruler itself the problem?**
That probe fit 200 trees at learning rate 0.1, because every fit in this project does,
on every dataset, always.

→ `phase5_model_capacity.ipynb`

In [ ]:
shutil.rmtree(os.path.dirname(run_dir), ignore_errors=True)